# Lab 6, part B: the session lifecycle

Lab 6 (A+B) budget: $0.30, or nothing beyond a Claude subscription

**What happens when one piece of work outlasts the window it started in.**

Part A chose routes. Part B is compaction, a summary that has to survive it, a resume after
somebody else has changed a file, two forks from one baseline, and the case where starting
fresh beats resuming.

The right-hand column of the diagram, with "evidence survives" running underneath it.

## 1. Where a session actually lives

**A session is a transcript on disk, not a memory.**

That single fact is what makes the rest of this part make sense: a resumed session reloads
what was said, not what the files now contain, so it can confidently tell you about a line
that moved last week.

Claude Code keys transcripts by working directory, with every character that is not a letter
or a digit turned into a hyphen, so the working copy you built in part A has its own folder.
A named session also writes its name there.

![Lab 6: code generation](../diagrams/lab-06-code-generation.png)


In [ ]:
import labkit

lab = labkit.start(model_env=None, credential="none")

import mycorp_lab

REPO, git = mycorp_lab.open_repo(lab)
print(f"working copy: {REPO}")
mycorp_lab.transcripts(REPO)

## 2. Two kinds of memory, side by side

**The project ships both, and they are not the same thing.**

`notes/investigation.md` is **prose**. A person re-reads it, and a session re-reads it when
the window no longer holds what it learned. Its discipline is that every line carries
something exact: a path, a line number, an amount, a date, a status.

`state/manifest.json` is **not prose**. It records which phase is done, which files that
phase actually read, and the commit the run started from. That shape is what lets a later
question be answered by arithmetic rather than by judgement: a finding stands if none of the
files it rests on has changed since. Section 7 does exactly that sum.

In [ ]:
mycorp_lab.show_memory(REPO)

## 3. One long session, compacted on purpose

Start a named session in the working copy:

```bash
cd workspace/mycorp
claude -n tkt-0031-duplicate-charge
```

`-n` names the session. The exam's answer for continuing a named investigation is
`--resume <session-name>`. On the CLI measured for this lab, `--resume` takes a session id
and a name is used as a search term that filters the picker, so you pick it from the list
rather than resolving it directly. `-r` is the short form of `--resume`; `-c` is
`--continue`, which is a different thing and takes no name. Section 1 printed the ids.

### One. Fill the window, so there is something to lose

> Read every module under `shop/` and give me one line each saying what it does and what it
> depends on.

### Two. The investigation

> Read `docs/TKT-0031-duplicate-charge.md`, then `docs/adr/ADR-0005-payment-gateway-retries.md`,
> then `shop/refunds/client.py` and `shop/payments/gateway.py`. What would it take to close
> TKT-0031? I want the mechanism, not a patch: where the idempotency key would have to be
> created, and what the retry in `send_refund` presents to the gateway on its second attempt.

### Three. Write it down as you go

> Keep `notes/investigation.md` current as you work. Every line has to carry something exact:
> a path, a line number, an amount, a date, a status, a decision, or an open question. A line
> I would have to go and look up again is a line that was not worth writing.

### Four. Close the phase before you open the next

> Summarise this phase in eight lines or fewer: what is settled, what is still open, and
> which files carry the evidence. Then update `state/manifest.json`: set `baseline_commit` to
> the output of `git rev-parse HEAD`, set the phases you have finished to `done`, put the
> files you actually read in each one's `files_read`, and put the summary in its `summary`.
> Keep the shape exactly as it is.

### Five. Compact, with direction

```
/compact Keep the TKT-0031 facts exactly: the amount, the date of the charge and the date of
the reversal, the ticket status, the file paths, the idempotency decision and the open
questions. Drop the module-by-module summaries and the file listings.
```

**Bare `/compact` is the failure mode.** Compaction replaces the conversation with a summary
and deletes the messages behind it, so what you type after the command is your only say in
what survives.

### Six. The survival questions

> What amount was charged twice, on what date, and on what date was the duplicate reversed?
> What is the status of TKT-0031, and which file holds the retry that caused it?

**Exact numbers, dates and statuses are the first thing a summary rounds off.** The cell below
prints the same facts from disk, so you can see whether they survived.

## 4. What survived

Hold the session's answers against the files. If one came back vague, **that is the lesson
landing rather than the lab failing**: the fix is a scratchpad the next session reads.

In [ ]:
print("the facts, from the ticket:")
for label, value in mycorp_lab.ticket_facts(REPO):
    print(f"  {label:10} {value}")

## 5. Somebody changed a file while you were away

**Nothing tells a resumed session that this happened.** It reloads a transcript, and the
transcript still holds a `Read` of the old file.

The cell below is the colleague. It folds the two `retry_count = 3` locals in
`shop/refunds/client.py` into one module constant and commits it, the way a teammate's push
would arrive while your session was closed.

In [ ]:
mycorp_lab.colleague_commit(REPO, git)

print("\nchanged since lab-start:")
for path in sorted(git.changed("lab-start")):
    print(f"  {path}")

## 6. Resume, fork, fresh

**Three ways back into a piece of work, and they are not interchangeable.**

### One. Resume, and name what changed

```bash
claude --resume
```

Pick `tkt-0031-duplicate-charge` from the picker, or pass the id section 1 printed. Then:

> While this session was closed, `shop/refunds/client.py` changed: the two retry counts are
> now one module constant, `RETRY_COUNT`. Nothing else moved. Re-read only that file, then
> tell me whether the idempotency finding in `notes/investigation.md` still holds, and which
> line the retry bound lives on now.

**Naming the file turns a full re-exploration into one `Read`.** Watch the tool calls: one
`Read`, not thirteen. That sentence is the whole technique.

### Two. Two forks from one baseline

```bash
claude --resume <id> --fork-session
```

> Take the design where the gateway client owns the idempotency key for the whole attempt
> sequence. Write it to `notes/fork-a-gateway-owns-key.md` under three headings: the change,
> the failure mode, what a test would assert. No code.

Then again, from the same session id:

```bash
claude --resume <id> --fork-session
```

> Take the design where the caller creates the idempotency key and passes it in. Same three
> headings, to `notes/fork-b-caller-owns-key.md`. No code.

`--fork-session` gives each branch a new session id instead of continuing the original, so
both start from the same investigation and neither disturbs it. **The comparison is the
deliverable, not the code.**

### Three. Fresh, with a summary you injected

By now much of the original transcript is reads of files at an older commit. **Resuming
carries the stale results along with the good ones, and nothing marks which is which.**

```bash
claude -n tkt-0031-decision
```

> Here is where the TKT-0031 investigation reached.
>
>     [paste the summary from state/manifest.json, and the decisions and open questions
>      from notes/investigation.md]
>
> One file has changed since that work: `shop/refunds/client.py`. Do not re-explore. Read
> that one file, then tell me which of the two idempotency designs to take, and why.

Decisions, identifiers, constraints, open questions. **Nothing about how they were found.**

### Four

Run `/cost` before you leave each session, and write the figure down.

## 7. Stale by arithmetic, not by feel

**Which findings still stand is not a judgement call, and it is not something to ask a model
that has just told you it remembers.**

It is an intersection: the files a phase read, against the files that have changed since the
commit it started from.

That is the whole reason the manifest records `files_read` and `baseline_commit` rather than
just a summary. **A scratchpad tells you what was learned; the manifest tells you what is
still true.**

In [ ]:
mycorp_lab.stale_phases(REPO, git)

## What you built

| The diagram | Where it happened |
|---|---|
| The repository, and its CLAUDE.md | Part A, sections 1 and 2 |
| Direct execution, for a small stack-trace fix | Part A, section 6, step one |
| Plan mode, for a multi-file migration and an undecided design | Part A, section 6, steps two and three |
| Explore subagent, read-only discovery | Part A, section 6, steps four and five |
| Tests first, two commits | Part A, sections 7 to 9 |
| Context compaction | Part B, section 3 |
| Durable summary | Part B, sections 2 and 4 |
| Resume | Part B, section 6, step one |
| Fork, and fresh session | Part B, section 6, steps two and three |
| Evidence survives | The git history, the scratchpad and the manifest |

**The decision to carry out of here is two sentences long.** Given a task and a codebase, say
whether to plan first or start typing, and say it before you type. Given a session that has
outlived its window, say whether to resume it or start fresh with a summary, and say what has
to be written down because the window will not keep it.